# 股票预测 - 探索性数据分析

本文档用于数据探索和模型分析。

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 设置显示
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# 图表设置
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

print('环境准备完成')

In [ ]:
# 数据库路径
LABELS_DB = Path('../data/labels.db')
FEATURES_DB = Path('../data/features.db')
SCREENER_DB = Path('../data/screener_scores.db')
PREDICTIONS_DB = Path('../data/predictions.db')

## 1. 标签分析

In [ ]:
# 加载标签数据
with sqlite3.connect(str(LABELS_DB)) as conn:
    labels = pd.read_sql_query("SELECT * FROM training_labels", conn)

print(f"标签数据量: {len(labels)}")
labels.head()

In [ ]:
# 标签分布
label_dist = labels['label'].value_counts()
print("标签分布:")
print(label_dist)
print(f"\n正样本率: {label_dist.get(1, 0) / len(labels) * 100:.2f}%")

In [ ]:
# 最大涨幅分布
plt.figure(figsize=(10, 5))
plt.hist(labels['max_gain'], bins=50, edgecolor='black')
plt.axvline(0.20, color='red', linestyle='--', label='目标 20%')
plt.xlabel('最大涨幅')
plt.ylabel('股票数量')
plt.title('最大涨幅分布')
plt.legend()
plt.show()

## 2. 特征分析

In [ ]:
# 加载特征数据
with sqlite3.connect(str(FEATURES_DB)) as conn:
    features = pd.read_sql_query("SELECT * FROM stock_features", conn)

print(f"特征数据量: {len(features)}")
features.head()

In [ ]:
# 特征缺失率
missing = features.isnull().sum() / len(features) * 100
missing = missing[missing > 0].sort_values(ascending=False)

print("特征缺失率 (>0%):")
print(missing.head(20))

In [ ]:
# RSI 分布
plt.figure(figsize=(10, 5))
plt.hist(features['rsi'].dropna(), bins=50, edgecolor='black')
plt.axvline(30, color='green', linestyle='--', label='超卖 (30)')
plt.axvline(70, color='red', linestyle='--', label='超买 (70)')
plt.xlabel('RSI')
plt.ylabel('股票数量')
plt.title('RSI 指标分布')
plt.legend()
plt.show()

## 3. 筛选器特征分析

In [ ]:
# 加载筛选器数据
with sqlite3.connect(str(SCREENER_DB)) as conn:
    screener = pd.read_sql_query("SELECT * FROM screener_features", conn)

print(f"筛选器数据量: {len(screener)}")
screener.head()

In [ ]:
# 各筛选器命中率
hit_rate = screener.groupby('screener_name')['hit'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
hit_rate.plot(kind='bar')
plt.ylabel('命中率')
plt.title('各筛选器命中率')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("筛选器命中率:")
print(hit_rate)

## 4. 预测结果分析

In [ ]:
# 加载预测结果
if PREDICTIONS_DB.exists():
    with sqlite3.connect(str(PREDICTIONS_DB)) as conn:
        predictions = pd.read_sql_query("SELECT * FROM predictions", conn)
    
    print(f"预测结果量: {len(predictions)}")
    predictions.head()
else:
    print("暂无预测结果")